# 9-9 網格防護與動態模擬：哨兵加框與雙矩陣快照（APCS f313 專題）

- **單元編號**：9-9
- **學習目標**：
  1. 理解「哨兵加框法（Grid Padding / Sentinel Border）」的設計思想，在網格四周加入保護外牆以徹底消除重複的邊界檢查。
  2. 掌握加框矩陣與原始矩陣的座標映射對應（$(r, c) \leftrightarrow (r+1, c+1)$），學會安全組裝與去框提取。
  3. 深刻洞悉二維網格動態模擬中，原地修改導致的「資料污染（Data Pollution）致命陷阱」。
  4. 熟練建構雙矩陣快照切換（Double Buffering）與增減量矩陣（Delta Grid）架構，確保狀態同步推進。
  5. 實作單回合相鄰擴散與結算邏輯，掌握四方向資源分配與淨變化量統計。
  6. 完整貫通 APCS 經典二級實作真題 **f313 人口遷移**，在無自訂函式架構下完成多回合動態擴散模擬與極值統計。
- **適合對象**：程式設計初學者（完全零基礎） / APCS 扎根學習者
- **先備知識**：9-1 二維陣列座標概念、9-3 記憶體獨立深拷貝、9-4 二維走訪與極值、9-7 方向向量與相鄰探測

---


### 9.9.1 哨兵加框法概念（Grid Padding / 邊界外圍加保護層）

在前面幾個單元（如 9-7、9-8）中，我們每次探測鄰居，都必須嚴格撰寫：
```python
if 0 <= nr < R and 0 <= nc < C:
    # 存取 grid[nr][nc]
```
雖然這種防護非常安全，但在某些需要連續模擬數十萬回合、運算極為密集的演算法中，每一次存取都要做 4 次比較運算（大於等於 0、小於 R、大於等於 0、小於 C），在多重巢狀迴圈中會累積相當大的效能開銷，且程式碼顯得較為繁重。

有沒有一種方法，可以**徹底不用寫任何邊界條件**，就能放膽探測周圍呢？
答案就是競賽中極具智慧的高階技巧——「**哨兵加框法（Grid Padding）**」！

#### 🧱 什麼是加框（Padding）？
想像一個 $R \times C$ 的原始房間。我們在房間的四周外圍，整整包上一圈**寬度為 1 的防護外牆（Sentinel Border）**：
- 最上方加一橫列
- 最下方加一橫列
- 最左側加一直行
- 最右側加一直行

於是，加框後的全新矩陣尺寸變成了 **$(R + 2) \times (C + 2)$**！
外牆通常填入特殊的「無效標記」（例如 `-1`、`None` 或 `0`）。
此時，原本在邊緣角落的格子，它的四個相鄰格子現在全都變成了加框矩陣中的「合法座標」！探測時絕不可能觸發 `IndexError`，我們只需要檢查「鄰居數值是否為外牆標記」，完全免去了冗長的數值邊界判斷！


In [ ]:
# 範例 9.9.1：為一個 2x3 的原始矩陣加上 -1 哨兵防護外框

raw_grid = [
    [10, 20, 30],
    [40, 50, 60]
]

R = len(raw_grid)
C = len(raw_grid[0])

print(f"原始矩陣尺寸：{R} x {C}")
for row in raw_grid:
    print(*row)

# 建立尺寸為 (R+2) x (C+2) 的加框矩陣，初始值全設為哨兵 -1
padded_grid = [[-1] * (C + 2) for _ in range(R + 2)]

# 將原始矩陣資料填入中央核心區域 (列 1 到 R，行 1 到 C)
for r in range(R):
    for c in range(C):
        padded_grid[r + 1][c + 1] = raw_grid[r][c]

print()
print(f"加框後矩陣尺寸：{R + 2} x {C + 2}")
for row in padded_grid:
    print(*row)


In [ ]:
# 填空 9.9.1：建構 3x3 網格的 0 哨兵保護外牆
# 請將 ___ 替換為正確的尺寸變數

A = [
    [1, 2, 3],
    [4, 5, 6],
    [7, 8, 9]
]

R = len(A)
C = len(A[0])

# 加框後的高度為 R + 2，寬度為 C + 2
padded = [[0] * (C + ___) for _ in range(R + ___)]

for r in range(R):
    for c in range(C):
        padded[r + 1][c + 1] = A[r][c]

print("加框後第一列（全為哨兵 0）：", padded[0])
print("加框後中央核心第一列：", padded[1])


In [ ]:
# 練習 9.9.1：外牆包覆器
# 題目說明：輸入兩個整數 R, C 與 R x C 整數矩陣。
# 請將該矩陣四周包上一圈哨兵外牆（外牆數值固定為 99），輸出加框後的整個矩陣（共 R+2 列，每列 C+2 個數字，以空白分隔）。

# 【公開測試資料 1】
# 2 2
# 5 6
# 7 8
# 輸出：
# 99 99 99 99
# 99 5 6 99
# 99 7 8 99
# 99 99 99 99

# 【公開測試資料 2】
# 1 3
# 1 2 3
# 輸出：
# 99 99 99 99 99
# 99 1 2 3 99
# 99 99 99 99 99

# 請在此處撰寫你的程式碼：
R, C = map(int, input().split())
grid = []
for _ in range(R):
    grid.append(list(map(int, input().split())))

padded = [[99] * (C + 2) for _ in range(R + 2)]
for r in range(R):
    for c in range(C):
        padded[r + 1][c + 1] = grid[r][c]

for row in padded:
    print(*row)


In [ ]:
# 挑戰 9.9.1：哨兵免邊界走訪測試
# 題目說明：輸入 R, C 與 R x C 整數矩陣。
# 將其加上一圈 -1 哨兵外牆。接著輸入原始座標 r, c (0 <= r < R, 0 <= c < C)。
# 在加框矩陣中，該點座標對應為 (r+1, c+1)。
# 請直接探測其四個相鄰格子（無需檢查 0 <= nr < R）：
# 若鄰居數值 != -1，代表為真實資料格，將其數值印出。
# 本題無公開測試資料，請自行驗證角落位置。

# 請在此處撰寫你的程式碼：
R, C = map(int, input().split())
grid = []
for _ in range(R):
    grid.append(list(map(int, input().split())))

r, c = map(int, input().split())

padded = [[-1] * (C + 2) for _ in range(R + 2)]
for i in range(R):
    for j in range(C):
        padded[i + 1][j + 1] = grid[i][j]

pr, pc = r + 1, c + 1
dr = [-1, 1, 0, 0]
dc = [0, 0, -1, 1]

valid_vals = []
for d in range(4):
    nr = pr + dr[d]
    nc = pc + dc[d]
    val = padded[nr][nc]
    if val != -1:
        valid_vals.append(val)

print(*valid_vals)


### 9.9.2 加框矩陣與原矩陣座標對應（$(r, c) \leftrightarrow (r+1, c+1)$）

加框法帶來了極大的防護便利，但它同時引入了一個初學者容易混淆的幾何映射——**座標偏移（Coordinate Offset）**！

讓我們清楚梳理兩個世界之間的座標關係：
1. **原始世界（Original Grid）**：
   - 列索引範圍：$0 \le r < R$
   - 行索引範圍：$0 \le c < C$
   - 左上角：`(0, 0)`，右下角：`(R-1, C-1)`
2. **加框世界（Padded Grid）**：
   - 尺寸：$(R + 2) \times (C + 2)$
   - 第 0 列、第 $R + 1$ 列是上下外牆
   - 第 0 行、第 $C + 1$ 行是左右外牆
   - **核心真實資料的走訪範圍**：
     - 列從 $1$ 走到 $R$：`for pr in range(1, R + 1):`
     - 行從 $1$ 走到 $C$：`for pc in range(1, C + 1):`

#### 🔄 雙向映射公式：
- **從原座標換算至加框座標**：$(pr, pc) = (r + 1, c + 1)$
- **從加框座標還原回原座標**：$(r, c) = (pr - 1, pc - 1)$

在完成所有模擬運算後，如果題目要求輸出原始網格，我們有兩種俐落的去框寫法：
- 寫法 A（巢狀迴圈）：`for pr in range(1, R + 1):` 依序輸出 `padded[pr][1 : C + 1]`。
- 寫法 B（切片生成）：`clean_grid = [row[1 : C + 1] for row in padded[1 : R + 1]]`。


In [ ]:
# 範例 9.9.2：加框矩陣走訪真實核心，並在運算後完美剝離外牆

padded = [
    [-1, -1, -1, -1],
    [-1, 10, 20, -1],
    [-1, 30, 40, -1],
    [-1, -1, -1, -1]
]

R = 2  # 原列數
C = 2  # 原行數

print("加框矩陣核心資料走訪（列 1~2, 行 1~2）：")
for pr in range(1, R + 1):
    for pc in range(1, C + 1):
        orig_r = pr - 1
        orig_c = pc - 1
        print(f"加框座標 ({pr}, {pc}) 對應原座標 ({orig_r}, {orig_c}) = {padded[pr][pc]}")

# 剝除外牆還原回乾淨的 2x2 矩陣
restored_grid = []
for pr in range(1, R + 1):
    # 切片取出該列去除頭尾外牆的部分
    core_row = padded[pr][1 : C + 1]
    restored_grid.append(core_row)

print()
print("去框還原後的乾淨矩陣：")
for row in restored_grid:
    print(*row)


In [ ]:
# 填空 9.9.2：只對加框矩陣的核心區域各加 5，並剝除外牆輸出
# 請將 ___ 替換為正確的走訪範圍

padded_board = [
    [0,  0,  0, 0],
    [0,  1,  2, 0],
    [0,  3,  4, 0],
    [0,  0,  0, 0]
]

R, C = 2, 2

# 走訪核心區域：列從 1 到 R，行從 1 到 C
for pr in range(1, ___ + 1):
    for pc in range(1, ___ + 1):
        padded_board[pr][pc] += 5

# 輸出核心區域
for pr in range(1, R + 1):
    print(*padded_board[pr][1 : C + 1])


In [ ]:
# 練習 9.9.2：加框去框流水線
# 題目說明：輸入兩個整數 R, C 與 R x C 整數矩陣。
# 請將其加上外牆 0，接著走訪加框矩陣核心區域，將每個核心格數值乘上 2。
# 最後將加框外牆去除，輸出變更後的原始尺寸矩陣。

# 【公開測試資料 1】
# 2 3
# 1 2 3
# 4 5 6
# 輸出：
# 2 4 6
# 8 10 12

# 【公開測試資料 2】
# 1 2
# 7 9
# 輸出：
# 14 18

# 請在此處撰寫你的程式碼：
R, C = map(int, input().split())
grid = []
for _ in range(R):
    grid.append(list(map(int, input().split())))

padded = [[0] * (C + 2) for _ in range(R + 2)]
for r in range(R):
    for c in range(C):
        padded[r + 1][c + 1] = grid[r][c]

for pr in range(1, R + 1):
    for pc in range(1, C + 1):
        padded[pr][pc] *= 2

for pr in range(1, R + 1):
    print(*(padded[pr][1 : C + 1]))


In [ ]:
# 挑戰 9.9.2：外牆哨兵安全性檢查
# 題目說明：給定一個已經加框的矩陣（尺寸為 H x W）。
# 請驗證其四周外圍的格子是否「全部為 0」（即第 0 列、第 H-1 列、第 0 行、第 W-1 行）。
# 若外框完好無損輸出 "Border Intact"，否則輸出 "Border Damaged"。
# 本題無公開測試資料，請自行測試。

# 請在此處撰寫你的程式碼：
H, W = map(int, input().split())
mat = []
for _ in range(H):
    mat.append(list(map(int, input().split())))

intact = True
for r in range(H):
    for c in range(W):
        if r == 0 or r == H - 1 or c == 0 or c == W - 1:
            if mat[r][c] != 0:
                intact = False
                break
    if not intact:
        break

if intact:
    print("Border Intact")
else:
    print("Border Damaged")


### 9.9.3 原地修改的「資料污染」致命陷阱（In-place Modification Pitfall）

在動態模擬題（例如：人口移動、溫度傳播、水流擴散、生命遊戲）中，題目通常會描述：
「**在同一個時間點（回合 $t$），所有格子同時向周圍擴散……**」

這句話隱含了一個電腦模擬中最容易翻車的致命陷阱——**時間同步性（Simultaneity）**！

#### 💥 慘烈的資料污染演示：
假設有兩個相鄰的城鎮 A 和 B，人口分別是：
- A: 100 人，B: 100 人
- 規則：每回合每個城鎮將自己一半的人口送給對方。
- **正確理論結果**：
  - A 送出 50，收到 B 的 50 $\rightarrow$ A 依然是 100
  - B 送出 50，收到 A 的 50 $\rightarrow$ B 依然是 100

現在看初學者的「邊走訪邊原地修改」會發生什麼悲劇：
1. 迴圈先走到 A：
   - A 送出 50 給 B，A 剩下 50，B 變成 $100 + 50 = 150$ 人！
2. 迴圈接著走到 B：
   - 此時 B 看到的人口已經變成 **150 人**了（被污染了）！
   - B 於是將 $150 // 2 = 75$ 人送給 A，B 剩下 75 人，A 變成 $50 + 75 = 125$ 人！
3. **錯誤結果**：A 變成 125，B 變成 75！整個系統崩潰了！

**致命教訓**：
在多格子互相影響的動態模擬中，**絕對嚴禁「一邊讀取舊狀態，一邊直接改寫當前矩陣」**！
所有格子的流動計算，都必須以「**回合開始時的凍結快照**」為基準，計算出的變動量必須另外存放，直到全圖所有格子都算完後，才能統一結算！


In [ ]:
# 範例 9.9.3：對比「原地修改的錯誤」與「快照計算的正確結果」

# 初始陣列：[100, 100]
print("--- 錯誤的原地修改（資料污染）---")
wrong_data = [100, 100]
# 假設規則：第 0 格將一半送給第 1 格，第 1 格將一半送給第 0 格
# 走訪第 0 格：
give_0 = wrong_data[0] // 2
wrong_data[0] -= give_0
wrong_data[1] += give_0  # 此時第 1 格被污染成 150！

# 走訪第 1 格：
give_1 = wrong_data[1] // 2  # 算出了 75，錯誤！
wrong_data[1] -= give_1
wrong_data[0] += give_1
print("錯誤結果：", wrong_data)  # [125, 75]

print()
print("--- 正確的快照運算（無污染）---")
correct_data = [100, 100]
# 建立凍結快照
snapshot = correct_data.copy()

# 嚴格依據快照算送出量
give_0 = snapshot[0] // 2  # 50
give_1 = snapshot[1] // 2  # 50

# 統一結算
correct_data[0] = snapshot[0] - give_0 + give_1
correct_data[1] = snapshot[1] - give_1 + give_0
print("正確結果：", correct_data)  # [100, 100]


In [ ]:
# 填空 9.9.3：使用深拷貝快照避免二維資料污染
# 請將 ___ 替換為正確的快照宣告

grid = [
    [10, 20],
    [30, 40]
]

# 宣告一張與 grid 完全獨立的舊狀態快照
snapshot = [row.___() for row in grid]

# 假設所有格子都變成相鄰四格的平均值，計算時必須嚴格讀取 snapshot
# 此處以 (0, 0) 為例，其新值參考 snapshot 的 (0, 1) 與 (1, 0)
new_val_00 = (snapshot[0][1] + snapshot[1][0]) // 2

# 寫入 grid 時絕不影響其他格子的參考依據
grid[0][0] = new_val_00

print("更新後 grid[0][0]：", grid[0][0])
print("快照 snapshot[0][0] 依然保持原值：", snapshot[0][0])


In [ ]:
# 練習 9.9.3：一維擴散無污染檢驗器
# 題目說明：給定一個長度為 N 的一維整數串列。
# 每回合每個格子會將自身整數數值的 10%（即 val // 10）送給右邊鄰居（最後一格送給右邊會出界，故不送）。
# 請使用「快照備份法」，計算一回合後每個格子的數值並輸出。

# 【公開測試資料 1】
# 3
# 100 100 100
# 輸出：90 100 110
# (第0格送10剩90；第1格送10收10剩100；第2格不送收10變110)

# 【公開測試資料 2】
# 2
# 50 0
# 輸出：45 5
# (第0格送5剩45；第1格收5變5)

# 請在此處撰寫你的程式碼：
N = int(input())
arr = list(map(int, input().split()))

snapshot = arr.copy()
res = arr.copy()

for i in range(N - 1):
    give = snapshot[i] // 10
    res[i] -= give
    res[i + 1] += give

print(*res)


In [ ]:
# 挑戰 9.9.3：污染偵測分析儀
# 題目說明：輸入長度 N 的串列。
# 請分別以「錯誤的原地修改」與「正確的快照法」執行一次練習 9.9.3 的擴散運算。
# 比較兩者最後的結果陣列：
# 若兩者完全相同輸出 "Same"，否則輸出 "Contaminated"。
# 本題無公開測試資料，請自行驗證。

# 請在此處撰寫你的程式碼：
N = int(input())
arr = list(map(int, input().split()))

# 原地修改
wrong = arr.copy()
for i in range(N - 1):
    give = wrong[i] // 10
    wrong[i] -= give
    wrong[i + 1] += give

# 快照法
correct = arr.copy()
snapshot = arr.copy()
for i in range(N - 1):
    give = snapshot[i] // 10
    correct[i] -= give
    correct[i + 1] += give

if wrong == correct:
    print("Same")
else:
    print("Contaminated")


### 9.9.4 增減量矩陣架構（Delta Grid）

在清楚了資料污染的嚴重性後，我們如何用最優美、最省記憶體且最不易出錯的方式實作網格模擬呢？
在演算法競賽（特別是 **APCS f313 人口遷移**）中，業界最推崇的標準架構是——「**增減量矩陣（Delta Grid）**」！

#### 📐 Delta 矩陣的運作哲學：
想像全圖各個城市在同時轉帳：
我們不去直接改動城市金庫，而是另外發給每座城市一張「**收支變動登記表（Delta Grid）**」：
1. **初始化登記表**：
   每回合開始時，建立一張與地圖尺寸完全相同、全為 0 的變動量矩陣：
   `delta = [[0] * C for _ in range(R)]`
2. **只登記變動量，不動原始數據**：
   - 當城市 A `(r, c)` 要將 $X$ 人送給相鄰城市 B `(nr, nc)` 時：
     - A 的登記表扣款：`delta[r][c] -= X`
     - B 的登記表入帳：`delta[nr][nc] += X`
   - 此時原始的 `grid` 完全沒有被碰觸，依然保持回合開始時的原貌！後續其他城市在參考 A 或 B 時，讀取到的永遠是未受污染的純淨原始數據！
3. **回合末統一結算（Final Settlement）**：
   當全圖所有格子的流動全部登記完畢後，執行一次全圖雙重迴圈，將登記表的淨變化量直接加回原始矩陣：
   ```python
   for r in range(R):
       for c in range(C):
           grid[r][c] += delta[r][c]
   ```
這種「只記變化量，最後統結算」的設計，徹底斬斷了資料污染的根源！


In [ ]:
# 範例 9.9.4：使用 Delta 矩陣完成 2x2 網格的向右流動結算

grid = [
    [10, 20],
    [30, 40]
]

R, C = 2, 2
print("初始狀態：")
for row in grid:
    print(*row)

# 1. 建立當前這回合的 delta 增減量矩陣
delta = [[0] * C for _ in range(R)]

# 假設規則：左邊一列將 5 單位移交給右邊一列
# (0, 0) 移給 (0, 1)；(1, 0) 移給 (1, 1)
amount = 5
delta[0][0] -= amount
delta[0][1] += amount

delta[1][0] -= amount
delta[1][1] += amount

print()
print("這回合的收支登記表 (Delta)：")
for row in delta:
    print(*row)

# 2. 統一結算加回原矩陣
for r in range(R):
    for c in range(C):
        grid[r][c] += delta[r][c]

print()
print("結算後的最終狀態：")
for row in grid:
    print(*row)


In [ ]:
# 填空 9.9.4：補齊 Delta 矩陣結算程式碼
# 請將 ___ 替換為正確的矩陣變數

P = [
    [100, 200],
    [300, 400]
]

R, C = 2, 2
# 建立全 0 增減量表
delta = [[0] * C for _ in range(R)]

# 登記流動：(0, 0) 扣 20，(1, 1) 加 20
delta[0][0] -= 20
delta[1][1] += 20

# 結算回 P
for r in range(R):
    for c in range(C):
        P[r][c] += ___[r][c]

print("P 結算結果：")
for row in P:
    print(*row)


In [ ]:
# 練習 9.9.4：多點匯聚 Delta 結算器
# 題目說明：輸入一個 3x3 整數矩陣。
# 建立一個全為 0 的 delta 矩陣。
# 規則：四周四個相鄰點 (0,1), (2,1), (1,0), (1,2) 各自將其數值的 10%（即 val // 10）匯入正中央 (1, 1)。
# 請使用 Delta 矩陣完成扣款與匯入登記，最後統一結算並輸出 3x3 矩陣。

# 【公開測試資料 1】
# 0 50 0
# 40 100 60
# 0 70 0
# 輸出：
# 0 45 0
# 36 122 54
# 0 63 0
# (四鄰居流出：50//10=5, 40//10=4, 60//10=6, 70//10=7，總匯入 5+4+6+7=22，中央 100+22=122)

# 【公開測試資料 2】
# 0 10 0
# 10 0 10
# 0 10 0
# 輸出：
# 0 9 0
# 9 4 9
# 0 9 0

# 請在此處撰寫你的程式碼：
grid = []
for _ in range(3):
    grid.append(list(map(int, input().split())))

delta = [[0] * 3 for _ in range(3)]
center_r, center_c = 1, 1

sources = [(0, 1), (2, 1), (1, 0), (1, 2)]
for r, c in sources:
    flow = grid[r][c] // 10
    delta[r][c] -= flow
    delta[center_r][center_c] += flow

for r in range(3):
    for c in range(3):
        grid[r][c] += delta[r][c]

for row in grid:
    print(*row)


In [ ]:
# 挑戰 9.9.4：質量守恆驗證員
# 題目說明：輸入 3x3 整數矩陣。
# 計算流動前矩陣所有元素的總和 sum1。
# 執行練習 9.9.4 的流動結算後，計算結算後的總和 sum2。
# 根據物理封閉系統質量守恆定律，sum1 應等於 sum2。
# 若相等輸出 "Conserved: " 加上總和，否則輸出 "Leaked"。
# 本題無公開測試資料，請自行測試。

# 請在此處撰寫你的程式碼：
grid = []
for _ in range(3):
    grid.append(list(map(int, input().split())))

sum1 = sum(sum(row) for row in grid)

delta = [[0] * 3 for _ in range(3)]
sources = [(0, 1), (2, 1), (1, 0), (1, 2)]
for r, c in sources:
    flow = grid[r][c] // 10
    delta[r][c] -= flow
    delta[1][1] += flow

for r in range(3):
    for c in range(3):
        grid[r][c] += delta[r][c]

sum2 = sum(sum(row) for row in grid)

if sum1 == sum2:
    print(f"Conserved: {sum1}")
else:
    print("Leaked")


### 9.9.5 單回合相鄰擴散與增減量結算

掌握了 Delta 矩陣後，我們現在把「方向向量探測」與「Delta 收支登記」融會貫通，實作一個完整的**單回合四方向相鄰擴散演算法**！

#### 🌊 四方向擴散的具體流程：
走訪地圖上的每一個格子 $(r, c)$：
1. **檢查自身是否有資源流出**：
   若當前格子的數值 $val \le 0$（例如人口為 0 或負數），則不具備流出能力，直接略過。
2. **探測合法接收鄰居**：
   使用方向向量 `dr, dc` 檢視上下左右四個鄰居 $(nr, nc)$：
   - 鄰居必須在邊界內：`0 <= nr < R and 0 <= nc < C`
   - 鄰居必須是有效格子（例如在 f313 中，`-1` 代表山脈空地，不能住人，不可接收人口）。
   - 統計出**合法鄰居的總個數 `valid_count`**，並記錄這些鄰居的座標。
3. **計算分配份額並登記 Delta**：
   - 如果分母常數為 $k$，則每個合法鄰居分到的量為：
     $$move\_per\_dir = val // k$$
   - 當前格子總共流出：
     $$total\_out = move\_per\_dir \times valid\_count$$
   - 登記自身扣款：`delta[r][c] -= total_out`
   - 登記每個合法鄰居入帳：`delta[nr][nc] += move_per_dir`
4. **全圖結算**：所有格子都登記完成後，一次性加回原地圖！


In [ ]:
# 範例 9.9.5：單回合四方向人口擴散演算法實作（分母 k = 5）

grid = [
    [50, 0],
    [ 0, 0]
]

R, C = 2, 2
k = 5

print("回合開始前人口：")
for row in grid:
    print(*row)

delta = [[0] * C for _ in range(R)]
dr = [-1, 1, 0, 0]
dc = [0, 0, -1, 1]

for r in range(R):
    for c in range(C):
        val = grid[r][c]
        if val <= 0:
            continue
            
        move_per_dir = val // k  # 50 // 5 = 10
        if move_per_dir == 0:
            continue
            
        valid_neighbors = []
        for d in range(4):
            nr = r + dr[d]
            nc = c + dc[d]
            if 0 <= nr < R and 0 <= nc < C:
                valid_neighbors.append((nr, nc))
                
        # 登記收支
        out_total = move_per_dir * len(valid_neighbors)
        delta[r][c] -= out_total
        for nr, nc in valid_neighbors:
            delta[nr][nc] += move_per_dir

# 統一結算
for r in range(R):
    for c in range(C):
        grid[r][c] += delta[r][c]

print()
print("一回合擴散結算後人口：")
for row in grid:
    print(*row)
# (0,0) 流出 20 剩 30，(0,1) 與 (1,0) 各接收 10


In [ ]:
# 填空 9.9.5：補齊擴散計算與 Delta 登記
# 請將 ___ 替換為正確的運算或變數

val = 43
k = 10
# 往每個方向流出的量為 val // k
flow_per_dir = ___ // ___  # 4

valid_neighbors = [(0, 1), (1, 0)]  # 2 個鄰居
delta_r, delta_c = 0, 0

# 自身流出總量 = flow_per_dir * 鄰居數
total_out = flow_per_dir * len(valid_neighbors)
delta[delta_r][delta_c] -= ___

# 每個鄰居各加 flow_per_dir
for nr, nc in valid_neighbors:
    delta[nr][nc] += ___


In [ ]:
# 練習 9.9.5：單回合避難所擴散結算
# 題目說明：輸入兩個整數 R, C 與整數 k 代表擴散分母。
# 接著輸入 R x C 整數地圖。
# 地圖中的數值代表人口；若為 -1 代表障礙物（山脈），無法居住亦不可接收人口。
# 請執行一回合擴散：
# 所有人數大於 0 的格子，往合法四鄰居（在邊界內且 != -1）各送出 val // k 人。
# 輸出結算後的地圖（每列以空白分隔）。

# 【公開測試資料 1】
# 2 2 4
# 40 -1
# 0 0
# 輸出：
# 30 -1
# 10 0
# ((0,0)=40，合法鄰居只有下(1,0)，向其送出 40//4=10；右邊是 -1 不能送；(0,0)扣10變30，(1,0)加10變10)

# 【公開測試資料 2】
# 1 3 2
# 20 0 0
# 輸出：
# 10 10 0
# ((0,0)=20 送出 20//2=10 給右邊 (0,1))

# 請在此處撰寫你的程式碼：
R, C, k = map(int, input().split())
grid = []
for _ in range(R):
    grid.append(list(map(int, input().split())))

delta = [[0] * C for _ in range(R)]
dr = [-1, 1, 0, 0]
dc = [0, 0, -1, 1]

for r in range(R):
    for c in range(C):
        val = grid[r][c]
        if val <= 0:
            continue
        move = val // k
        if move == 0:
            continue
        valid_neighbors = []
        for d in range(4):
            nr = r + dr[d]
            nc = c + dc[d]
            if 0 <= nr < R and 0 <= nc < C and grid[nr][nc] != -1:
                valid_neighbors.append((nr, nc))
        delta[r][c] -= move * len(valid_neighbors)
        for nr, nc in valid_neighbors:
            delta[nr][nc] += move

for r in range(R):
    for c in range(C):
        grid[r][c] += delta[r][c]

for row in grid:
    print(*row)


In [ ]:
# 挑戰 9.9.5：多障礙物孤島零擴散檢驗
# 題目說明：輸入 R, C, k 與 R x C 地圖。
# 若某個格子的四個鄰居全部都是 -1（或出界），該格子無法向任何人擴散。
# 請執行一回合擴散後，統計全圖共有幾個人口數 > 0 的格子其數值「完全沒有發生任何變動」。
# 本題無公開測試資料，請自行測試。

# 請在此處撰寫你的程式碼：
R, C, k = map(int, input().split())
grid = []
for _ in range(R):
    grid.append(list(map(int, input().split())))

orig = [row.copy() for row in grid]
delta = [[0] * C for _ in range(R)]
dr = [-1, 1, 0, 0]
dc = [0, 0, -1, 1]

for r in range(R):
    for c in range(C):
        val = grid[r][c]
        if val <= 0:
            continue
        move = val // k
        if move == 0:
            continue
        valid = []
        for d in range(4):
            nr = r + dr[d]
            nc = c + dc[d]
            if 0 <= nr < R and 0 <= nc < C and grid[nr][nc] != -1:
                valid.append((nr, nc))
        delta[r][c] -= move * len(valid)
        for nr, nc in valid:
            delta[nr][nc] += move

unchanged = 0
for r in range(R):
    for c in range(C):
        grid[r][c] += delta[r][c]
        if orig[r][c] > 0 and grid[r][c] == orig[r][c]:
            unchanged += 1

print(unchanged)


### 9.9.6 APCS f313 人口遷移：多回合動態擴散模擬實戰（APCS 實戰原型）

恭喜你！我們抵達了第九章（二維陣列）最壯麗的終點站——**APCS 實作真題 f313 人口遷移**！

#### 📜 完整題意規則拆解：
給定一個 $R \times C$ 的城鎮地圖：
- 每個格子代表一個城鎮的人口數。
- 若格子數值為 `-1`，代表該地為無法居住的山脈或湖泊，**不能住人，也不能接收任何流動人口**。
- 給定擴散係數 $k$ 與模擬回合數 $m$。
- 在每一回合中：
  - 所有人數大於 0 的城鎮，向其上下左右**可以居住（非 -1 且在邊界內）**的鄰近城鎮各遷移 $\lfloor P / k \rfloor$ 人。
  - 遷移是**同時發生**的，必須使用 Delta 增減量矩陣防禦資料污染。
- 總共模擬 $m$ 回合。
- **最終目標**：
  在 $m$ 回合結束後，找出全圖所有城鎮（**排除 -1 的格子**）中，**人口最多的城鎮人數** 與 **人口最少的城鎮人數**，並依序輸出這兩個數值！

#### 🏆 無自訂函式的最高工程組織：
在完全不依賴 `def` 的純粹 Python 結構下，我們將程式組織為清晰的三大板塊：
1. **輸入與初值準備**：讀取 $R, C, k, m$ 與人口地圖。
2. **多回合大迴圈（`for round in range(m):`）**：
   - 每回合新建歸零的 `delta` 矩陣。
   - 雙重走訪計算並登記各城鎮流出與鄰居流入。
   - 統一結算加回 `grid`。
3. **全圖極值檢索**：走訪結算後的 `grid`，跳過 `-1`，更新 `min_pop` 與 `max_pop`，最後印出答案！
這道題目集結了二維串列、向量走訪、邊界防護、資料快照與多回合狀態推進的所有智慧，是 APCS 二級分乃至三級分的最高榮譽殿堂！


In [ ]:
# 範例 9.9.6：APCS f313 多回合人口遷移完整演示

R, C, k, m = 3, 3, 5, 2  # 3x3 地圖，分母 5，模擬 2 回合
grid = [
    [ 0, 50,  0],
    [-1,  0, 50],
    [ 0,  0,  0]
]

print("初始人口地圖：")
for row in grid:
    print(*row)

dr = [-1, 1, 0, 0]
dc = [0, 0, -1, 1]

# 執行 m 回合模擬
for round_num in range(1, m + 1):
    delta = [[0] * C for _ in range(R)]
    
    for r in range(R):
        for c in range(C):
            val = grid[r][c]
            if val <= 0:
                continue
            move = val // k
            if move == 0:
                continue
                
            valid_neighbors = []
            for d in range(4):
                nr = r + dr[d]
                nc = c + dc[d]
                if 0 <= nr < R and 0 <= nc < C and grid[nr][nc] != -1:
                    valid_neighbors.append((nr, nc))
                    
            delta[r][c] -= move * len(valid_neighbors)
            for nr, nc in valid_neighbors:
                delta[nr][nc] += move
                
    # 結算
    for r in range(R):
        for c in range(C):
            grid[r][c] += delta[r][c]

print()
print(f"經過 {m} 回合模擬後的人口：")
for row in grid:
    print(*row)

# 統計非 -1 的極小值與極大值
min_pop = None
max_pop = None

for r in range(R):
    for c in range(C):
        if grid[r][c] != -1:
            val = grid[r][c]
            if min_pop is None or val < min_pop:
                min_pop = val
            if max_pop is None or val > max_pop:
                max_pop = val

print()
print(f"最少人口：{min_pop}，最多人口：{max_pop}")


In [ ]:
# 填空 9.9.6：補齊 APCS f313 排除 -1 的極值搜尋邏輯
# 請將 ___ 替換為正確的條件或變數

min_val = None
max_val = None

for r in range(R):
    for c in range(C):
        # 題目規定：排除 -1 的障礙空格
        if grid[r][c] != ___:
            pop = grid[r][c]
            if min_val is None or pop < min_val:
                min_val = pop
            if max_val is None or pop > ___:
                max_val = pop

print(min_val, max_val)


In [ ]:
# 練習 9.9.6：APCS f313 人口遷移真題實戰
# 題目說明：
# 第一行輸入四個整數 R, C, k, m (1 <= R, C <= 50, 1 <= k <= 10, 1 <= m <= 50)。
# 接下來有 R 行，每行包含 C 個整數，代表初始人口數（-1 代表不可居住）。
# 請模擬 m 回合的人口遷移，最後在同一行輸出最少人口數與最多人口數（以空白分隔）。

# 【公開測試資料 1】
# 3 3 5 2
# 0 50 0
# -1 0 50
# 0 0 0
# 輸出：1 26
# (模擬 2 回合後，扣除 -1 區域，全圖最少城鎮人口為 1，最多為 26)

# 【公開測試資料 2】
# 2 2 10 1
# 100 -1
# -1 50
# 輸出：50 100
# (兩城鎮相隔 -1，無法互相流動，人數保持 50 與 100)

# 請在此處撰寫你的程式碼：
R, C, k, m = map(int, input().split())
grid = []
for _ in range(R):
    grid.append(list(map(int, input().split())))

dr = [-1, 1, 0, 0]
dc = [0, 0, -1, 1]

for _ in range(m):
    delta = [[0] * C for _ in range(R)]
    for r in range(R):
        for c in range(C):
            val = grid[r][c]
            if val <= 0:
                continue
            move = val // k
            if move == 0:
                continue
            valid = []
            for d in range(4):
                nr = r + dr[d]
                nc = c + dc[d]
                if 0 <= nr < R and 0 <= nc < C and grid[nr][nc] != -1:
                    valid.append((nr, nc))
            delta[r][c] -= move * len(valid)
            for nr, nc in valid:
                delta[nr][nc] += move

    for r in range(R):
        for c in range(C):
            grid[r][c] += delta[r][c]

min_p = None
max_p = None
for r in range(R):
    for c in range(C):
        if grid[r][c] != -1:
            v = grid[r][c]
            if min_p is None or v < min_p:
                min_p = v
            if max_p is None or v > max_p:
                max_p = v

print(min_p, max_p)


In [ ]:
# 挑戰 9.9.6：人口穩定收斂回合檢測器
# 題目說明：在 f313 的規則下，有時經過多次遷移後，人口會達到「動態平衡（某一回合的 delta 全為 0）」。
# 請撰寫程式：最多模擬 m 回合，但若在第 t 回合結束時發現「全圖沒有任何人口流動」，
# 則提前結束模擬，並印出收斂的回合數 t；若 m 回合結束皆未收斂，輸出 -1。
# 本題無公開測試資料，請自行測試。

# 請在此處撰寫你的程式碼：
R, C, k, m = map(int, input().split())
grid = []
for _ in range(R):
    grid.append(list(map(int, input().split())))

dr = [-1, 1, 0, 0]
dc = [0, 0, -1, 1]

converged_round = -1

for t in range(1, m + 1):
    delta = [[0] * C for _ in range(R)]
    any_movement = False
    
    for r in range(R):
        for c in range(C):
            val = grid[r][c]
            if val <= 0:
                continue
            move = val // k
            if move == 0:
                continue
            valid = []
            for d in range(4):
                nr = r + dr[d]
                nc = c + dc[d]
                if 0 <= nr < R and 0 <= nc < C and grid[nr][nc] != -1:
                    valid.append((nr, nc))
            if valid:
                any_movement = True
                delta[r][c] -= move * len(valid)
                for nr, nc in valid:
                    delta[nr][nc] += move

    for r in range(R):
        for c in range(C):
            grid[r][c] += delta[r][c]
            
    if not any_movement:
        converged_round = t
        break

print(converged_round)


## 🎯 第九章（二維陣列）全景通關總結與榮譽認證

恭喜你！歷經 **9 大核心專題、54 個微型學習階梯** 的深耕特訓，你已經完整通關了 Python 程式設計中最具分水嶺意義的**第九章 二維陣列（2D Array / List of Lists）**！

### 🗺️ 第九章 9 節全景通關全紀錄：
- 🟢 **9-1 二維陣列概念與座標元素存取**：建立列主序、型態辨析與座標定位。
- 🟢 **9-2 二維陣列動態輸入讀取與解包輸出**：掌握迴圈輸入、列表生成式與解包格式化排版。
- 🟢 **9-3 二維陣列初始化與參照共用致命陷阱**：破解淺拷貝淺拷貝慘劇，掌握獨立深拷貝與布林標記。
- 🟢 **9-4 二維網格雙重走訪與行列統計**：精通時鐘模型、橫列直接統計、直行固定縱向累加與極值鎖定。
- 🟢 **9-5 二維方陣與特殊走訪：對角線與棋盤規律**：主副對角線單層極速走訪、上下三角與棋盤染色。
- 🟢 **9-6 矩陣幾何操作與逆推還原（APCS b266 專題）**：翻轉、旋轉、轉置、動態維度追蹤與逆推還原。
- 🟢 **9-7 二維網格導航：方向向量與相鄰探測（APCS e287 原型）**：方向向量 `dr, dc`、邊界防護與貪婪尋路。
- 🟢 **9-8 網格射線掃描（Raycasting）與連線阻擋（APCS g596 專題）**：步進 `while`、障礙阻擋與連線重繪。
- 🟢 **9-9 網格防護與動態模擬：哨兵加框與雙矩陣快照（APCS f313 專題）**：哨兵加框、資料污染防禦與 Delta 矩陣。

---
🏆 **榮譽授予【二維網格大師徽章】🏆**！你已經具備在 APCS 實作考場中從容攻克任何二維矩陣考題的絕對實力！接下來，我們將邁向**第十章 集合（Set）與元組（Tuple）**，開啟更高效的資料結構新紀元！
